# Process Pool Executor
The `ProcessPoolExecutor` class is the most modern and easiest way to convert a for-loop to run in parallel for CPU-bound tasks.

Process pools are a design pattern that let you execute and manage heterogeneous, discrete, and ad hoc tasks.

In [1]:
from random import random
from time import sleep
from multiprocessing import Process
from concurrent.futures import ProcessPoolExecutor, TimeoutError, wait, as_completed, FIRST_COMPLETED, FIRST_EXCEPTION

## Processes, Executors, and Process Pools
**1. What is a process-based concurrency**

A process is a computer program.

Every Python program is a process and has one thread called the main thread used to execute our program instructions. Each process is, in fact, one instance of a Python interpreter that executes Python bytecode called the `MainProcess`

**2. Process pools provide reusable workers**

A process pool is a programming pattern for automatically managing a pool of worker processes. Responsible for:
* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

The `concurrent.futures` module was introduced by Brian Quinlan.

The `ProcessPoolExecutor` extends the `Executor` class and will return `Future` objects when it is called.
* `Executor`: Parent class for the `ProcessPoolExecutor` that defines basic life-cycle operations for the pool.
* `Future`: Object returned when submitting tasks to the process pool that may complete later.

The `Executor` class defines three methods used to control our process pool:
* `submit()`: Dispatch a function to be executed and return a `Future` object. 
* `map()`: Call a function for each item in a iterable. Each application to the function to an element will happen concurrently instead of sequentially.
* `shutdown()`: Shut down the `Executor`.

A `Future` is an object that represents a delayed result for an asynchronous task. Sometimes called a `promise` or `delay`. It provides a context for the result of a task that may or may not be executing and a way of getting a result once it is available.

A `Future` object is returned from an `Executor` when calling the `submit()` method to dispatch a task to be executed asynchronously.

`Future` object methods for inspecting the status of the task:
* `cancelled()`: Returns `True` if the task was canceled before being executed. A running task cannot be canceled.  
* `running()`: Returns `True` if the task is currently running.
* `done()`: Returns `True` if the task has completed or was canceled.

`Future` object methods for accessing the `result()` method. Both allow a timeout to be specified as an argument. If the timeout expires, then a `TimeoutError` will be raised:
* `result()`: Access the result from running the task.
* `exception()`: Access any exception raised while running the task.

Automatically call a function once the task is completed:
* `add_done_callback()`: Add a callback function to the task to be executed by the process pool once the task is completed.

**3. How to use the `ProcessPoolExecutor`**

`ProcessPoolExecutor` three main steps in the life-cycle:
1. Create: Create the process pool by calling the constructor `ProcessPoolExecutor()`.
2. Execute: Run tasks using workers via the `map()` or `submit()` methods.   
3. Shut Down: Shut down the process pool by calling `shutdown()`.

## Configure the `ProcessPoolExecutor`
**1. Configure the `ProcessPoolExecutor`**

Arguments:
* `max_workers`: Maximum number of worker processes to use in the pool.
* `mp_context`: The multiprocessing context is used to create worker processes.
* `initializer`: Function executed after each worker process is created.
* `initargs`: Arguments to the worker process initialization function.

**2. Configure the number of worker processes.**

Default Worker Processes = min(61, Logical CPUs)

```python
# protect the entry point
if __name__ == '__main__':
    # create a process pool
    exe = ProcessPoolExecutor()
    # report the status of the process pool
    print(exe._max_workers)
    # shutdown the process pool
    exe.shutdown()
```
Configure it yourself:
```python
# create a process pool with 4 workers
exe = ProcessPoolExecutor(max_workers=4)
```
There is no upper limit on the number of processes that can be created on most platforms. The exception is Windows that limits the maximum number of processes to 61.

If we are expecting to perform computational work in the main process in addition to the multiprocessing pool, consider setting the number of processes in the pool to be equal to the number of logical CPUs in our system minus one, to allow the main process to execute.

If we have particularly CPU intensive tasks, consider configuring the number of processes to be equal to the number of physical CPUs instead of the number of logical CPUs.

In [ ]:
# custom task function executed in the process pool
def task(number):
    # block for a moment
    sleep(1)
    # report a message
    if number % 10 == 0:
        print(f"> task {number} done", flush=True)
        
# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor(50) as exe:
        # issue many tasks to the pool
        _ = exe.map(task, range(50))

**3. Configure the start method.**

Three start methods:
* `spawn`: start a new Python process. Default on Windows and MacOS.
* `fork`: copy a Python process from an existing process. Not supported in Windows.
* `forkserver`: new process from which future forked processes will be copied.

```python
# create a new context with the spawn start method
ctx = get_context('spawn')
# create a process pool with a given context
exe = ProcessPoolExecutor(mp_context=ctx)
```

**4. Configure the worker initializer function.**

May be helpful for worker processes to prepare resources that may be used across the execution of many tasks, such as logging infrastructure or a result queue.
```python
# create a process pool and initialize workers
exe = ProcessPoolExecutor(initializer=init, initargs=(a1, a2))
```

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # report a message
    print(f"Worker task {number}...", flush=True)
    # block for a moment
    sleep(1)
    
# initialize a worker in a process pool
def init():
    # report a message
    print("Initializing worker ...", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create and configure the process pool
    with ProcessPoolExecutor(2, initializer=init) as exe:
        # issue tasks to the process pool
        _ = exe.map(task, range(4))

## Execute Multiple Tasks Concurrently
**1. What is the `map()` method**

**2. Issue many tasks with one or multiple arguments.**

**3. Issue many tasks with no return values.**

**4. Issue many tasks and use a timeout when getting results.**

**5. Optimize the execution of many tasks by executing them in chunks.**


## Execute One-Off Tasks Asynchronously
**1. What the `submit()` method is.**

**2. Issue ad hoc tasks asynchronously with multiple arguments.**

**3. Issue ad hoc tasks asynchronously with no return values.**

**4. Issue multiple ad hoc tasks asynchronously systematically.**

## Query Asynchronously Tasks
**1. What are `Future` objects**

**2. Check the status of `Future` objects**

**3. Get results from `Future` objects**

**4. Cancel `Future` objects**

**5. Add callbakcs to `Future` objects**

**6. Get exceptions from `Future` objects**

## Manage Collections of Asynchronously Tasks
**1. What are the module functions**

**2. Handle results as tasks finish**

**3. Wait for all tasks**

**4. Wait for the first task**

**5. Wait for the first task failure**



## Case Study: Calculate Fibonacci Numbers
**Slow**

**Fast**